In [1]:
# ===========================
# Phase III modeling: Base vs Cluster, Regressor + Classifier
# ===========================

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, roc_auc_score, average_precision_score, classification_report
from lightgbm import LGBMRegressor, LGBMClassifier

# ---------------------------
# 0. Load data (already engineered)
# ---------------------------
df = pd.read_csv("./patient_features_with_encounter_cluster.csv")

# Ensure types
df['encounter_start_dt'] = pd.to_datetime(df['encounter_start_dt'], errors='coerce')
df['encounter_end_dt']   = pd.to_datetime(df['encounter_end_dt'],   errors='coerce')


In [2]:
# ---------------------------
# 1. Targets
# ---------------------------
# A) Duration regression target (unchanged)
target_days = 'encounter_duration_days'

# B) Count target (choose one)
#    Option 1 (your original): past_num_encounters (historical)
target_count = 'past_num_encounters'
#    Option 2 (optional): current encounter procedure count
# target_count = 'proc_count'  # uncomment to switch

# C) Short vs Long classifier target
#    Define long-stay threshold in days (e.g., >= 3 days is long)
long_stay_threshold = 3.0
df['is_long_stay'] = (df[target_days] >= long_stay_threshold).astype(int)


In [3]:
# ---------------------------
# 2. Feature sets
#    Remove encounter_setting; keep encounter_medical_service (categorical).
#    Cluster version adds 'encounter_cluster'.
# ---------------------------
base_features = [
    'age', 'PATIENT_SEX', 'PATIENT_RACE_ETHNICITY',
    'encounter_medical_service',
    'past_num_encounters', 'past_total_procedures',
    'past_total_icd_codes', 'past_total_encounter_days'
    # DO NOT include encounter_setting
]

cluster_features = base_features + ['encounter_cluster']

# Sanity: keep only columns that exist
base_features    = [c for c in base_features    if c in df.columns]
cluster_features = [c for c in cluster_features if c in df.columns]


In [4]:
# ---------------------------
# 3. Train/Test split with GroupKFold (by patient)
#    We first create a single holdout split to report metrics consistently.
# ---------------------------
# Create groups
groups = df['patient_id'].astype(str)

# Single split (80/20) preserving groups
# To use GroupKFold for a single split, we can manually take the first fold as test
gkf = GroupKFold(n_splits=5)
train_idx, test_idx = next(gkf.split(df, groups=groups))

df_train = df.iloc[train_idx].reset_index(drop=True)
df_test  = df.iloc[test_idx].reset_index(drop=True)

def build_preproc(feature_list):
    # separate categorical and numeric
    cat_cols = [c for c in feature_list if df[c].dtype == 'object']
    num_cols = [c for c in feature_list if c not in cat_cols]

    preproc = ColumnTransformer(
        transformers=[
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
            ('num', 'passthrough', num_cols)
        ],
        remainder='drop'
    )
    return preproc, cat_cols, num_cols

def fit_and_eval_reg(feature_list, y_col, label):
    preproc, cat_cols, num_cols = build_preproc(feature_list)
    model = LGBMRegressor(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )
    pipe = Pipeline([
        ('prep', preproc),
        ('lgbm', model)
    ])
    X_tr, y_tr = df_train[feature_list], df_train[y_col]
    X_te, y_te = df_test[feature_list],  df_test[y_col]

    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)

    mae = mean_absolute_error(y_te, pred)
    r2  = r2_score(y_te, pred)
    print(f"[{label}] Regression {y_col}: MAE={mae:.3f}, R²={r2:.3f}")
    return pipe, pred, {'MAE': mae, 'R2': r2}

def fit_and_eval_cls(feature_list, y_col, label):
    preproc, cat_cols, num_cols = build_preproc(feature_list)
    clf = LGBMClassifier(
        n_estimators=600,
        learning_rate=0.05,
        num_leaves=63,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced',
        random_state=42
    )
    pipe = Pipeline([
        ('prep', preproc),
        ('lgbm', clf)
    ])
    X_tr, y_tr = df_train[feature_list], df_train[y_col]
    X_te, y_te = df_test[feature_list],  df_test[y_col]

    pipe.fit(X_tr, y_tr)
    proba = pipe.predict_proba(X_te)[:,1]
    pred  = (proba >= 0.5).astype(int)

    auc = roc_auc_score(y_te, proba)
    ap  = average_precision_score(y_te, proba)
    print(f"[{label}] Classifier {y_col}: AUC={auc:.3f}, PR-AUC={ap:.3f}")
    print(classification_report(y_te, pred, digits=3))
    return pipe, proba, {'AUC': auc, 'PR_AUC': ap}
    

In [5]:
# ---------------------------
# 4. Train & evaluate (Base vs Cluster)
# ---------------------------
print("Data shape:", df.shape)

# Regressors
base_reg_count, base_pred_count, base_count_m = fit_and_eval_reg(base_features, target_count,  "Base")
base_reg_days,  base_pred_days,  base_days_m  = fit_and_eval_reg(base_features, target_days,   "Base")

cluster_reg_count, cluster_pred_count, cluster_count_m = fit_and_eval_reg(cluster_features, target_count, "Cluster")
cluster_reg_days,  cluster_pred_days,  cluster_days_m  = fit_and_eval_reg(cluster_features, target_days,  "Cluster")

# Classifier (short vs long)
base_cls,    base_prob_long,    base_cls_m    = fit_and_eval_cls(base_features,   'is_long_stay', "Base")
cluster_cls, cluster_prob_long, cluster_cls_m = fit_and_eval_cls(cluster_features,'is_long_stay', "Cluster")


Data shape: (15116, 27)
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000814 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 315
[LightGBM] [Info] Number of data points in the train set: 12092, number of used features: 30
[LightGBM] [Info] Start training from score 0.740655
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


C:\Users\liuzi\anaconda3\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[Base] Regression past_num_encounters: MAE=0.207, R²=0.634
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000486 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 315
[LightGBM] [Info] Number of data points in the train set: 12092, number of used features: 30
[LightGBM] [Info] Start training from score 9.708180


C:\Users\liuzi\anaconda3\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[Base] Regression encounter_duration_days: MAE=9.195, R²=0.085
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000791 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 320
[LightGBM] [Info] Number of data points in the train set: 12092, number of used features: 31
[LightGBM] [Info] Start training from score 0.740655
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


...

C:\Users\liuzi\anaconda3\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[Cluster] Regression encounter_duration_days: MAE=9.208, R²=0.088
[LightGBM] [Info] Number of positive: 6489, number of negative: 5603
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000521 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 315
[LightGBM] [Info] Number of data points in the train set: 12092, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


C:\Users\liuzi\anaconda3\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[Base] Classifier is_long_stay: AUC=0.810, PR-AUC=0.769
              precision    recall  f1-score   support

           0      0.856     0.614     0.715      1425
           1      0.725     0.908     0.806      1599

    accuracy                          0.770      3024
   macro avg      0.791     0.761     0.761      3024
weighted avg      0.787     0.770     0.763      3024

[LightGBM] [Info] Number of positive: 6489, number of negative: 5603
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000550 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 320
[LightGBM] [Info] Number of data points in the train set: 12092, number of used features: 31
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[Cluster] Classifier is_long_stay: AUC=0.808, PR-AUC=0.764
   

C:\Users\liuzi\anaconda3\lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
# ---------------------------
# 5. Service-level workload comparison (using predictions)
#    For days: use predicted duration; for count: use predicted count.
# ---------------------------
def agg_workload(df_sub, pred_count, pred_days):
    tmp = df_sub[['encounter_medical_service']].copy()
    tmp['pred_count'] = pred_count
    tmp['pred_days']  = pred_days
    grp = tmp.groupby('encounter_medical_service').agg(
        Patients=('pred_days','size'),
        Avg_pred_count=('pred_count','mean'),
        Avg_pred_days =('pred_days','mean')
    )
    grp['Total_workload_days'] = grp['Avg_pred_days'] * grp['Patients']
    return grp

base_work = agg_workload(df_test, base_pred_count,    base_pred_days)
clst_work = agg_workload(df_test, cluster_pred_count, cluster_pred_days)

comp = (base_work
        .join(clst_work, lsuffix="_base", rsuffix="_cluster")
        .assign(
            Delta_avg_days = lambda d: d['Avg_pred_days_cluster'] - d['Avg_pred_days_base'],
            Delta_total_days = lambda d: d['Total_workload_days_cluster'] - d['Total_workload_days_base']
        )
       ).sort_values('Total_workload_days_base', ascending=False)

print("\n=== Global metrics (test set) ===")
print("  Model    MAE_count   R2_count   MAE_days   R2_days   AUC(long)  PR-AUC(long)")
print(f"  Base     {base_count_m['MAE']:.3f}       {base_count_m['R2']:.3f}     {base_days_m['MAE']:.3f}    {base_days_m['R2']:.3f}     {base_cls_m['AUC']:.3f}      {base_cls_m['PR_AUC']:.3f}")
print(f"  Cluster  {cluster_count_m['MAE']:.3f}       {cluster_count_m['R2']:.3f}     {cluster_days_m['MAE']:.3f}    {cluster_days_m['R2']:.3f}     {cluster_cls_m['AUC']:.3f}      {cluster_cls_m['PR_AUC']:.3f}")

print("\n=== Workload comparison by medical service (Cluster vs Base) ===")
print(comp.reset_index().to_string(index=False))




=== Global metrics (test set) ===
  Model    MAE_count   R2_count   MAE_days   R2_days   AUC(long)  PR-AUC(long)
  Base     0.207       0.634     9.195    0.085     0.810      0.769
  Cluster  0.212       0.623     9.208    0.088     0.808      0.764

=== Workload comparison by medical service (Cluster vs Base) ===
           encounter_medical_service  Patients_base  Avg_pred_count_base  Avg_pred_days_base  Total_workload_days_base  Patients_cluster  Avg_pred_count_cluster  Avg_pred_days_cluster  Total_workload_days_cluster  Delta_avg_days  Delta_total_days
        Nursing - medical / surgical           1574             0.522406           13.760912              21659.676089              1574                0.521603              13.798640                 21719.059286        0.037728         59.383197
                     General surgery            184             0.413340            8.624613               1586.928747               184                0.413744               8.619234     

In [8]:
# ======================================================================
# ONE CELL: Do BOTH tasks (ICD features + DBSCAN model) and extend results
# Runs AFTER your Base/Cluster cell. Produces ICD-only & ICD+DBSCAN models,
# service-level workload tables, and merges them into your existing `comp`.
# ======================================================================

import numpy as np
import pandas as pd
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# ---------------------------
# 0) Expectation: df exists. If df_train/df_test are missing, create them.
# ---------------------------
if "df" not in globals():
    raise RuntimeError("This cell expects a DataFrame `df` already loaded.")

if "df_train" not in globals() or "df_test" not in globals():
    strat = df["is_long_stay"] if "is_long_stay" in df.columns else None
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=strat)

# ---------------------------
# 1) Targets: LOS required; COUNT optional. Auto-detect LOS or compute.
# ---------------------------
def _pick_col(cands, frame):
    lut = {c.lower(): c for c in frame.columns}
    for k in cands:
        if k in lut: return lut[k]
    return None

LOS_CANDS = ["encounter_los","los","length_of_stay","encounter_duration_days","encounter_days","stay_days"]
TARGET_DAYS = _pick_col([c.lower() for c in LOS_CANDS], df_train)

if TARGET_DAYS is None:
    need = {"encounter_start_dt","encounter_end_dt"}
    if need.issubset(df_train.columns) and need.issubset(df_test.columns):
        for d in (df_train, df_test):
            d["encounter_start_dt"] = pd.to_datetime(d["encounter_start_dt"], errors="coerce")
            d["encounter_end_dt"]   = pd.to_datetime(d["encounter_end_dt"],   errors="coerce")
            los = (d["encounter_end_dt"] - d["encounter_start_dt"]).dt.total_seconds()/86400.0
            d["encounter_los"] = los.clip(lower=0).fillna(0)
        TARGET_DAYS = "encounter_los"
    else:
        raise KeyError("No LOS column found and cannot compute LOS (need start/end).")
print(f"✅ Using LOS target: {TARGET_DAYS}")

COUNT_CANDS = ["future_encounter_count","encounter_count_next","count_target","predicted_count"]
TARGET_COUNT = _pick_col([c.lower() for c in COUNT_CANDS], df_train)
print("ℹ️ Count target:", TARGET_COUNT if TARGET_COUNT else "None (will use zeros in workload)")

# ---------------------------
# 2) TASK 1 — Add ICD top-K one-hot columns
# ---------------------------
ICD_COL  = "icd_prefix3_unique_str"  # pipe-delimited 3-char ICD prefixes
ICD_SEP  = "|"
ICD_TOPK = 30

if ICD_COL not in df_train.columns:
    raise KeyError(f"Missing ICD column '{ICD_COL}'.")

def _parse_icd(series, sep=ICD_SEP):
    return series.fillna("").astype(str).str.strip().apply(lambda s: [t for t in s.split(sep) if t])

train_icd_lists = _parse_icd(df_train[ICD_COL])
test_icd_lists  = _parse_icd(df_test[ICD_COL])

counts = Counter(p for lst in train_icd_lists for p in lst)
topk_icd = [p for p,_ in counts.most_common(ICD_TOPK)]

for p in topk_icd:
    col = f"icd_{p}"
    df_train[col] = train_icd_lists.apply(lambda lst: int(p in lst))
    df_test[col]  = test_icd_lists.apply(lambda lst: int(p in lst))

icd_features = [f"icd_{p}" for p in topk_icd]
print(f"✅ Added {len(icd_features)} ICD one-hot columns.")

# Base numeric features (use what exists)
DEFAULT_BASE_NUMERIC = ["age","past_num_encounters","past_total_procedures","past_total_icd_codes","past_total_encounter_days"]
BASE_NUMERIC = [c for c in DEFAULT_BASE_NUMERIC if c in df_train.columns]

# ---------------------------
# 3) ICD-only models (days required; count optional)
# ---------------------------
pre_icd = ColumnTransformer([
    ("num", StandardScaler(), BASE_NUMERIC),
    ("icd", "passthrough", icd_features),
])

rf_icd_days = Pipeline([
    ("pre", pre_icd),
    ("rf", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
])
rf_icd_days.fit(df_train[BASE_NUMERIC + icd_features], df_train[TARGET_DAYS])
icd_pred_days = rf_icd_days.predict(df_test[BASE_NUMERIC + icd_features])

if TARGET_COUNT:
    rf_icd_cnt = Pipeline([
        ("pre", pre_icd),
        ("rf", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
    ])
    rf_icd_cnt.fit(df_train[BASE_NUMERIC + icd_features], df_train[TARGET_COUNT])
    icd_pred_count = rf_icd_cnt.predict(df_test[BASE_NUMERIC + icd_features])
else:
    icd_pred_count = np.zeros(len(df_test))

icd_days_m = {"MAE": mean_absolute_error(df_test[TARGET_DAYS], icd_pred_days),
              "R2":  r2_score(df_test[TARGET_DAYS], icd_pred_days)}
if TARGET_COUNT:
    icd_count_m = {"MAE": mean_absolute_error(df_test[TARGET_COUNT], icd_pred_count),
                   "R2":  r2_score(df_test[TARGET_COUNT], icd_pred_count)}
else:
    icd_count_m = {"MAE": np.nan, "R2": np.nan}

# ---------------------------
# 4) TASK 2 — Add a DBSCAN model (cluster on train+test consistently)
# ---------------------------
cluster_feats = BASE_NUMERIC + icd_features
X_all = pd.concat([df_train[cluster_feats], df_test[cluster_feats]], axis=0).fillna(0.0)

scaler = StandardScaler()
X_all_scaled = scaler.fit_transform(X_all)

pca = PCA(n_components=min(20, X_all_scaled.shape[1]), random_state=42)
X_all_pca = pca.fit_transform(X_all_scaled)

dbs = DBSCAN(eps=0.8, min_samples=15)
labels_all = dbs.fit_predict(X_all_pca)

labels_train = labels_all[:len(df_train)]
labels_test  = labels_all[len(df_train):]

df_train["dbscan_cluster"] = labels_train
df_test["dbscan_cluster"]  = labels_test

# DBSCAN-augmented models
pre_icd_dbs = ColumnTransformer([
    ("num", StandardScaler(), BASE_NUMERIC),
    ("icd", "passthrough", icd_features),
    ("dbs", OneHotEncoder(handle_unknown="ignore"), ["dbscan_cluster"]),
])

rf_dbs_days = Pipeline([
    ("pre", pre_icd_dbs),
    ("rf", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
])
rf_dbs_days.fit(df_train[cluster_feats + ["dbscan_cluster"]], df_train[TARGET_DAYS])
dbs_pred_days = rf_dbs_days.predict(df_test[cluster_feats + ["dbscan_cluster"]])

if TARGET_COUNT:
    rf_dbs_cnt = Pipeline([
        ("pre", pre_icd_dbs),
        ("rf", RandomForestRegressor(n_estimators=400, random_state=42, n_jobs=-1))
    ])
    rf_dbs_cnt.fit(df_train[cluster_feats + ["dbscan_cluster"]], df_train[TARGET_COUNT])
    dbs_pred_count = rf_dbs_cnt.predict(df_test[cluster_feats + ["dbscan_cluster"]])
else:
    dbs_pred_count = np.zeros(len(df_test))

dbs_days_m = {"MAE": mean_absolute_error(df_test[TARGET_DAYS], dbs_pred_days),
              "R2":  r2_score(df_test[TARGET_DAYS], dbs_pred_days)}
if TARGET_COUNT:
    dbs_count_m = {"MAE": mean_absolute_error(df_test[TARGET_COUNT], dbs_pred_count),
                   "R2":  r2_score(df_test[TARGET_COUNT], dbs_pred_count)}
else:
    dbs_count_m = {"MAE": np.nan, "R2": np.nan}

# ---------------------------
# 5) Service-level workload (reuses your agg_workload)
# ---------------------------
if "agg_workload" not in globals():
    raise RuntimeError("This cell expects your agg_workload(df_sub, pred_count, pred_days) function to exist.")

icd_work = agg_workload(df_test, icd_pred_count, icd_pred_days)
dbs_work = agg_workload(df_test, dbs_pred_count, dbs_pred_days)

# ---------------------------
# 6) Merge with your existing `comp` (Base vs Cluster) safely with explicit suffixes
# ---------------------------
def _suffix(df_in, suf):
    return df_in.rename(columns={c: f"{c}{suf}" for c in df_in.columns})

icd_work_s = _suffix(icd_work, "_icd")
dbs_work_s = _suffix(dbs_work, "_dbscan")

if "comp" in globals():
    comp_ext = comp.join(icd_work_s, how="left").join(dbs_work_s, how="left")
    comp_ext = comp_ext.assign(
        Delta_avg_days_icd       = comp_ext["Avg_pred_days_icd"]       - comp_ext["Avg_pred_days_base"],
        Delta_total_days_icd     = comp_ext["Total_workload_days_icd"] - comp_ext["Total_workload_days_base"],
        Delta_avg_days_dbscan    = comp_ext["Avg_pred_days_dbscan"]    - comp_ext["Avg_pred_days_base"],
        Delta_total_days_dbscan  = comp_ext["Total_workload_days_dbscan"] - comp_ext["Total_workload_days_base"],
    ).sort_values("Total_workload_days_base", ascending=False)
else:
    comp_ext = None
    print("ℹ️ Note: `comp` not found, so only ICD/DBSCAN workload tables are printed.")

# ---------------------------
# 7) Print metrics + tables
# ---------------------------
print("\n=== Additional Global Metrics (test set) ===")
print("  Model     MAE_count      R2_count      MAE_days       R2_days")
def fmt(x): 
    return "NA" if (isinstance(x, float) and np.isnan(x)) or (x is None) else f"{x:.3f}"
print(f"  ICD       {fmt(icd_count_m['MAE']):>10}   {fmt(icd_count_m['R2']):>10}   {fmt(icd_days_m['MAE']):>10}   {fmt(icd_days_m['R2']):>10}")
print(f"  DBSCAN    {fmt(dbs_count_m['MAE']):>10}   {fmt(dbs_count_m['R2']):>10}   {fmt(dbs_days_m['MAE']):>10}   {fmt(dbs_days_m['R2']):>10}")

print("\n=== Workload by Medical Service — ICD only ===")
print(icd_work.reset_index().to_string(index=False))

print("\n=== Workload by Medical Service — ICD + DBSCAN ===")
print(dbs_work.reset_index().to_string(index=False))

if comp_ext is not None:
    print("\n=== Workload comparison by medical service (ICD & DBSCAN vs Base) — FINAL ===")
    print(comp_ext.reset_index().to_string(index=False))


✅ Using LOS target: encounter_duration_days
ℹ️ Count target: None (will use zeros in workload)
✅ Added 30 ICD one-hot columns.

=== Additional Global Metrics (test set) ===
  Model     MAE_count      R2_count      MAE_days       R2_days
  ICD               NA           NA       10.063        0.045
  DBSCAN            NA           NA       10.008        0.050

=== Workload by Medical Service — ICD only ===
           encounter_medical_service  Patients  Avg_pred_count  Avg_pred_days  Total_workload_days
                          Cardiology        55             0.0       8.494462           467.195407
             Cardiovascular services        12             0.0       9.136972           109.643662
                            Cath lab         5             0.0      11.829761            59.148804
              Critical care medicine        60             0.0       8.811373           528.682362
                  Emergency medicine       748             0.0       8.531579          6381.6213